<a href="https://colab.research.google.com/github/ToneCapone1123/Surface-Dwellers-Surface-Code-Decoder-/blob/main/SurfaceDecoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install stim numpy

In [ ]:
import numpy as np
import stim


class ParitySweepDecoder:
    """
    Classical row/column parity-sweep decoder.

    The inputs are cumulative region parities over a binary error grid.

    row_scan[i, j]:
        XOR of errors along row i, from column 0 through j.

    col_scan[i, j]:
        XOR of errors along column j, from row 0 through i.
    """

    def __init__(self, distance):
        self.d = distance

    @staticmethod
    def xor_difference(a, b):
        return np.bitwise_xor(a, b)

    def decode_rows(self, row_scan):
        """
        Recover errors from row-prefix parity measurements.
        """
        row_scan = np.asarray(row_scan, dtype=np.uint8)

        if row_scan.shape != (self.d, self.d):
            raise ValueError("Incorrect row scan dimensions")

        errors = np.zeros_like(row_scan)

        # First column: prefix parity equals the first error.
        errors[:, 0] = row_scan[:, 0]

        # Subsequent columns: parity changes indicate errors.
        errors[:, 1:] = (
            row_scan[:, 1:] ^ row_scan[:, :-1]
        )

        return errors

    def decode_columns(self, col_scan):
        """
        Recover errors from column-prefix parity measurements.
        """
        col_scan = np.asarray(col_scan, dtype=np.uint8)

        if col_scan.shape != (self.d, self.d):
            raise ValueError("Incorrect column scan dimensions")

        errors = np.zeros_like(col_scan)

        errors[0, :] = col_scan[0, :]

        errors[1:, :] = (
            col_scan[1:, :] ^ col_scan[:-1, :]
        )

        return errors

    def decode(self, row_scan, col_scan):
        """
        Recover error locations using both sweeps.

        Returns:
            row_errors: errors reconstructed from row scans
            col_errors: errors reconstructed from column scans
            consistent: whether both sweeps agree
        """
        row_errors = self.decode_rows(row_scan)
        col_errors = self.decode_columns(col_scan)

        consistent = np.array_equal(row_errors, col_errors)

        return {
            "row_errors": row_errors,
            "col_errors": col_errors,
            "consistent": consistent,
            "decoded_errors": (
                row_errors if consistent else None
            )
        }

    @staticmethod
    def prefix_xor_rows(errors):
        """
        Generate row-prefix parities from an error grid.
        """
        return np.bitwise_xor.accumulate(errors, axis=1)

    @staticmethod
    def prefix_xor_columns(errors):
        """
        Generate column-prefix parities from an error grid.
        """
        return np.bitwise_xor.accumulate(errors, axis=0)

In [ ]:
import numpy as np

d = 5
decoder = ParitySweepDecoder(distance=d)

# Example physical error configuration
errors = np.zeros((d, d), dtype=np.uint8)

errors[1, 2] = 1
errors[3, 0] = 1
errors[3, 4] = 1

print("Actual errors:")
print(errors)

# Simulate the two cumulative parity sweeps
row_scan = decoder.prefix_xor_rows(errors)
col_scan = decoder.prefix_xor_columns(errors)

print("\nRow scan:")
print(row_scan)

print("\nColumn scan:")
print(col_scan)

# Decode
result = decoder.decode(row_scan, col_scan)

print("\nReconstructed from rows:")
print(result["row_errors"])

print("\nReconstructed from columns:")
print(result["col_errors"])

print("\nSweeps consistent:", result["consistent"])

import numpy as np
import matplotlib.pyplot as plt


def plot_lattice(matrix, title="Lattice", cmap="Blues", annotate=True):
    """
    Plot a matrix as a square lattice.

    Each matrix entry corresponds to a lattice cell.
    Values are printed inside cells.
    """

    matrix = np.asarray(matrix)

    d = matrix.shape[0]

    fig, ax = plt.subplots(figsize=(7, 7))

    # Display matrix as a lattice
    ax.imshow(
        matrix,
        cmap=cmap,
        vmin=0,
        vmax=max(1, matrix.max()),
        origin="upper"
    )

    # Grid lines
    ax.set_xticks(np.arange(-0.5, d, 1))
    ax.set_yticks(np.arange(-0.5, d, 1))

    ax.grid(color="black", linewidth=1.5)

    # Tick labels
    ax.set_xticks(np.arange(d))
    ax.set_yticks(np.arange(d))

    ax.set_xlabel("Column j")
    ax.set_ylabel("Row i")

    ax.set_title(title)

    # Annotate each cell
    if annotate:
        for i in range(d):
            for j in range(d):
                ax.text(
                    j,
                    i,
                    str(matrix[i, j]),
                    ha="center",
                    va="center",
                    color="black",
                    fontsize=12
                )

    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()

Actual errors:
[[0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [1 0 0 0 1]
 [0 0 0 0 0]]

Row scan:
[[0 0 0 0 0]
 [0 0 1 1 1]
 [0 0 0 0 0]
 [1 1 1 1 0]
 [0 0 0 0 0]]

Column scan:
[[0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]
 [1 0 1 0 1]
 [1 0 1 0 1]]

Reconstructed from rows:
[[0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [1 0 0 0 1]
 [0 0 0 0 0]]

Reconstructed from columns:
[[0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [1 0 0 0 1]
 [0 0 0 0 0]]

Sweeps consistent: True


In [ ]:
import stim

d = 5
rounds = 5
p = 0.001

circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    distance=d,
    rounds=rounds,
    after_clifford_depolarization=p,
    before_measure_flip_probability=p,
    after_reset_flip_probability=p,
)

print(circuit)

# Sample measurement outcomes
sampler = circuit.compile_sampler(seed=42)

samples = sampler.sample(shots=1000)

print("Sample shape:", samples.shape)

QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(6, 0) 6
QUBIT_COORDS(7, 1) 7
QUBIT_COORDS(9, 1) 9
QUBIT_COORDS(1, 3) 12
QUBIT_COORDS(2, 2) 13
QUBIT_COORDS(3, 3) 14
QUBIT_COORDS(4, 2) 15
QUBIT_COORDS(5, 3) 16
QUBIT_COORDS(6, 2) 17
QUBIT_COORDS(7, 3) 18
QUBIT_COORDS(8, 2) 19
QUBIT_COORDS(9, 3) 20
QUBIT_COORDS(10, 2) 21
QUBIT_COORDS(0, 4) 22
QUBIT_COORDS(1, 5) 23
QUBIT_COORDS(2, 4) 24
QUBIT_COORDS(3, 5) 25
QUBIT_COORDS(4, 4) 26
QUBIT_COORDS(5, 5) 27
QUBIT_COORDS(6, 4) 28
QUBIT_COORDS(7, 5) 29
QUBIT_COORDS(8, 4) 30
QUBIT_COORDS(9, 5) 31
QUBIT_COORDS(1, 7) 34
QUBIT_COORDS(2, 6) 35
QUBIT_COORDS(3, 7) 36
QUBIT_COORDS(4, 6) 37
QUBIT_COORDS(5, 7) 38
QUBIT_COORDS(6, 6) 39
QUBIT_COORDS(7, 7) 40
QUBIT_COORDS(8, 6) 41
QUBIT_COORDS(9, 7) 42
QUBIT_COORDS(10, 6) 43
QUBIT_COORDS(0, 8) 44
QUBIT_COORDS(1, 9) 45
QUBIT_COORDS(2, 8) 46
QUBIT_COORDS(3, 9) 47
QUBIT_COORDS(4, 8) 48
QUBIT_COORDS(5, 9) 49
QUBIT_COORDS(6, 8) 50
QUBIT_COORDS(7, 9) 51
QUBIT_COORDS(8,